# N-grams Playground — coding workshop (beginner)

**Expected time:** ~15 minutes, individual, no GPU needed.

You just watched counting turn into probabilities on a tiny corpus about an aunt and an
uncle. Here you rebuild that exact model yourself, watch it generate text, and then push
it one step further: use it to fix the word order of a word-for-word translation.

## How this workshop works

This is the coding version of the notebook, **beginner level**. Some code cells have gaps
marked `___`. An exercise above each such cell says what to write. This version takes
longer than the expected time given above.

1. Read the exercise. Open the hints if you need them.
2. Replace every `___` in the cell with your code, then run the cell.
3. Run the check cell right below it. It prints `✓ Exercise N passed`, or an error
   message that says what to look at.
4. If you are stuck, the solution is below the check cell. Open it after you have tried.

A cell that still contains `___` stops with `NameError: name '___' is not defined`.
Later cells use your code, so finish each exercise before moving on.

## Part A — Build the bigram model

An n-gram is a sequence of *n* consecutive words. This is a **bigram** model: it predicts
each word using only the one word before it, by counting how often each pair of words
occurs together in a corpus and dividing.

### ✏️ Exercise 1 — Count bigrams and turn counts into probabilities

Complete `build_bigram_counts` and `bigram_prob`.

- `counts[w1][w2]` is how many times the pair `w1 w2` occurs in the corpus.
- `unigram_counts[w1]` is how many pairs start with `w1`.
- `bigram_prob` returns P(w2 | w1) = count(w1 w2) / count(w1).

When you run the cell, it should print the lesson's numbers: 0.67 and 0.33.

<details><summary><b>Hints</b> (click to open)</summary>

- The inner loop looks at position `i` and the word right after it, `words[i + 1]`.
- The pair count is indexed by the first word, then the second word.
- The probability divides the pair count by the count of the first word.

</details>

In [ ]:
from collections import defaultdict

CORPUS = [
    "my aunt stayed quiet",
    "my aunt spoke wisely",
    "my uncle stayed quiet",
]


def build_bigram_counts(corpus):
    counts = defaultdict(lambda: defaultdict(int))
    unigram_counts = defaultdict(int)
    for sentence in corpus:
        words = sentence.lower().split()
        for i in range(len(words) - 1):
            counts[words[i]][___] += 1
            unigram_counts[___] += 1
    return counts, unigram_counts


def bigram_prob(w1, w2, counts, unigram_counts):
    if unigram_counts[w1] == 0:
        return 0.0
    return ___ / ___


counts, unigram_counts = build_bigram_counts(CORPUS)

print(f"P(aunt | my)  = {bigram_prob('my', 'aunt', counts, unigram_counts):.3f}   (lesson: 0.67)")
print(f"P(uncle | my) = {bigram_prob('my', 'uncle', counts, unigram_counts):.3f}   (lesson: 0.33)")

In [ ]:
# Check for Exercise 1. Run this cell as is.
def test_example():
    """P(aunt | my) is 2/3 and P(uncle | my) is 1/3"""
    p_aunt = bigram_prob("my", "aunt", counts, unigram_counts)
    p_uncle = bigram_prob("my", "uncle", counts, unigram_counts)
    if p_aunt == 0 and counts["my"]["aunt"] == 2 and unigram_counts["my"] == 3:
        assert False, "P(aunt | my) is 0 although the counts are right. With // the division rounds down to a whole number: use /"
    assert abs(p_aunt - 2 / 3) < 1e-9, f"P(aunt | my) is {p_aunt}, expected 2/3 = 0.667. The tests below say which part is off"
    assert abs(p_uncle - 1 / 3) < 1e-9, f"P(uncle | my) is {p_uncle}, expected 1/3 = 0.333"


def test_pair_order():
    """a pair is counted as (first word, next word), in that order"""
    assert counts["my"]["my"] == 0, "counts['my']['my'] should be 0: the pair needs the NEXT word, words[i + 1]"
    assert counts["aunt"]["my"] == 0, "counts['aunt']['my'] should be 0: 'aunt my' never occurs. Are the two words swapped?"
    assert counts["my"]["aunt"] == 2, f"counts['my']['aunt'] is {counts['my']['aunt']}, expected 2"
    assert counts["stayed"]["quiet"] == 2, "'stayed quiet' occurs twice in the corpus"


def test_unigram_counts():
    """count(w1) counts the pairs that START with w1"""
    assert unigram_counts["my"] == 3, f"unigram_counts['my'] is {unigram_counts['my']}, expected 3: 'my' starts three pairs"
    assert unigram_counts["quiet"] == 0, "unigram_counts['quiet'] should be 0: a sentence's last word starts no pair. Count words[i], not words[i + 1]"


def test_probabilities_sum_to_one():
    """the probabilities of all words that can follow a word add up to 1"""
    for w1 in ["my", "aunt", "stayed"]:
        total = sum(bigram_prob(w1, w2, counts, unigram_counts) for w2 in list(counts[w1]))
        assert abs(total - 1) < 1e-9, f"after '{w1}' the probabilities add up to {total:.3f}, not 1. Check the counts above, and that the pair count is divided by count(w1)"


def test_unseen_words():
    """unseen words get probability 0, without an error"""
    assert bigram_prob("her", "aunt", counts, unigram_counts) == 0, "a first word never seen should give 0"
    assert bigram_prob("my", "cat", counts, unigram_counts) == 0, "a pair never seen should give 0"


def test_short_corpora():
    """a one-word sentence has no pairs, and neither has an empty corpus"""
    c, u = build_bigram_counts(["hello"])
    assert sum(u.values()) == 0, "a one-word sentence has no pairs"
    c, u = build_bigram_counts([])
    assert sum(u.values()) == 0, "an empty corpus has no pairs"


def test_repeated_word():
    """'a a a' has two pairs, both 'a a'"""
    c, u = build_bigram_counts(["a a a"])
    assert c["a"]["a"] == 2, f"counts['a']['a'] is {c['a']['a']}, expected 2"
    assert u["a"] == 2, f"unigram_counts['a'] is {u['a']}, expected 2: the last 'a' starts no pair"


def test_capital_letters():
    """'My' and 'my' are the same word"""
    c, u = build_bigram_counts(["My Aunt", "my aunt"])
    assert u["my"] == 2 or c["my"]["aunt"] == 2, "capital letters made a different word: keep sentence.lower()"

_failed = 0
for _test in [test_example, test_pair_order, test_unigram_counts, test_probabilities_sum_to_one, test_unseen_words, test_short_corpora, test_repeated_word, test_capital_letters]:
    try:
        _test()
        print("✓", _test.__doc__)
    except Exception as _e:
        _failed += 1
        print("✗", _test.__doc__)
        print("   ", _e if isinstance(_e, AssertionError) else f"{type(_e).__name__}: {_e}")
assert _failed == 0, f"{_failed} test(s) failed: see the ✗ lines above"
print("✓ Exercise 1 passed")

<details><summary><b>Solution to Exercise 1</b> (open it after you have tried)</summary>

```python
from collections import defaultdict

CORPUS = [
    "my aunt stayed quiet",
    "my aunt spoke wisely",
    "my uncle stayed quiet",
]


def build_bigram_counts(corpus):
    counts = defaultdict(lambda: defaultdict(int))
    unigram_counts = defaultdict(int)
    for sentence in corpus:
        words = sentence.lower().split()
        for i in range(len(words) - 1):
            counts[words[i]][words[i + 1]] += 1
            unigram_counts[words[i]] += 1
    return counts, unigram_counts


def bigram_prob(w1, w2, counts, unigram_counts):
    if unigram_counts[w1] == 0:
        return 0.0
    return counts[w1][w2] / unigram_counts[w1]


counts, unigram_counts = build_bigram_counts(CORPUS)

print(f"P(aunt | my)  = {bigram_prob('my', 'aunt', counts, unigram_counts):.3f}   (lesson: 0.67)")
print(f"P(uncle | my) = {bigram_prob('my', 'uncle', counts, unigram_counts):.3f}   (lesson: 0.33)")
```

</details>

## Try it yourself

Add 2-3 sentences of your own to the corpus below — reuse the same words (`aunt`,
`uncle`, `stayed`, `spoke`, `quiet`, `wisely`) or introduce new ones — then re-run and see
how the probabilities shift. This *is* the whole model: more data, different (usually
better) estimates.

### ✏️ Exercise 2 — Change the data, change the model

Add at least two sentences to `EXTENDED_CORPUS` so that, in the new model,
P(uncle | my) is **larger** than P(aunt | my). Then run the cell and check.

<details><summary><b>Hints</b> (click to open)</summary>

- The original corpus has 'my aunt' twice and 'my uncle' once.
- Each new sentence is a Python string, for example `"my uncle spoke kindly"`.

</details>

In [ ]:
EXTENDED_CORPUS = CORPUS + [
    ___,  # a sentence, as a string
    ___,  # another sentence
]

ext_counts, ext_unigrams = build_bigram_counts(EXTENDED_CORPUS)
print(f"P(aunt | my)  = {bigram_prob('my', 'aunt', ext_counts, ext_unigrams):.3f}")
print(f"P(uncle | my) = {bigram_prob('my', 'uncle', ext_counts, ext_unigrams):.3f}")

In [ ]:
# Check for Exercise 2. Run this cell as is.
def test_two_new_sentences():
    """at least two new sentences, each a separate string"""
    new = EXTENDED_CORPUS[len(CORPUS):]
    assert all(isinstance(s, str) for s in EXTENDED_CORPUS), "every sentence should be a string, in quotes"
    assert len(new) != 0 or len(CORPUS) == 3, "CORPUS itself was changed. See the last test"
    assert len(new) != 1, ("found 1 new sentence. A missing comma between two strings joins them into one: "
                           "\"my uncle\" \"my uncle\" is the single string \"my unclemy uncle\"")
    assert len(new) >= 2, "add at least two sentences after CORPUS"
    assert all(len(s.split()) >= 2 for s in new), "each new sentence needs at least two words to make a pair"


def test_uncle_wins():
    """in the new model, P(uncle | my) is larger than P(aunt | my)"""
    p_uncle = bigram_prob("my", "uncle", ext_counts, ext_unigrams)
    p_aunt = bigram_prob("my", "aunt", ext_counts, ext_unigrams)
    assert p_uncle != p_aunt, f"both are {p_uncle:.3f}: a tie. Add one more sentence with 'my uncle'"
    assert p_uncle > p_aunt, (f"P(uncle | my) = {p_uncle:.3f}, P(aunt | my) = {p_aunt:.3f}. "
                              "Only the pair 'my uncle' raises it: 'uncle' must come right after 'my'")


def test_original_model_unchanged():
    """the original corpus and model are unchanged, for the next exercises"""
    assert CORPUS == ["my aunt stayed quiet", "my aunt spoke wisely", "my uncle stayed quiet"], \
        "CORPUS was changed. Build EXTENDED_CORPUS as a new list with CORPUS + [...], and run Exercise 1 again"
    assert abs(bigram_prob("my", "aunt", counts, unigram_counts) - 2 / 3) < 1e-9, \
        "the original model changed. Use new names (ext_counts, ext_unigrams), and run Exercise 1 again"

_failed = 0
for _test in [test_two_new_sentences, test_uncle_wins, test_original_model_unchanged]:
    try:
        _test()
        print("✓", _test.__doc__)
    except Exception as _e:
        _failed += 1
        print("✗", _test.__doc__)
        print("   ", _e if isinstance(_e, AssertionError) else f"{type(_e).__name__}: {_e}")
assert _failed == 0, f"{_failed} test(s) failed: see the ✗ lines above"
print("✓ Exercise 2 passed")

<details><summary><b>Solution to Exercise 2</b> (open it after you have tried)</summary>

```python
EXTENDED_CORPUS = CORPUS + [
    "my uncle spoke kindly",
    "my uncle stayed calm",
]

ext_counts, ext_unigrams = build_bigram_counts(EXTENDED_CORPUS)
print(f"P(aunt | my)  = {bigram_prob('my', 'aunt', ext_counts, ext_unigrams):.3f}")
print(f"P(uncle | my) = {bigram_prob('my', 'uncle', ext_counts, ext_unigrams):.3f}")
```

</details>

## Generating text from the model

Given a start word, repeatedly pick whichever next word has the highest count — that's a
bigram language model actually *generating* text, going beyond scoring it.

### ✏️ Exercise 3 — Generate text greedily

Complete the loop in `generate`: at each step, pick the next word with the highest
count after the current word, and add it to the sentence.

<details><summary><b>Hints</b> (click to open)</summary>

- `candidates` maps each possible next word to its count, e.g. `{'aunt': 2, 'uncle': 1}`.
- `max(candidates, key=...)` returns the key with the largest value of `key(word)`.

</details>

In [ ]:
def generate(start_word, counts, unigram_counts, max_words=10):
    word = start_word
    sentence = [word]
    for _ in range(max_words):
        candidates = counts[word]
        if not candidates:
            break
        word = max(candidates, key=lambda w: ___)
        sentence.append(___)
    return " ".join(sentence)


print(generate("my", counts, unigram_counts))

In [ ]:
# Check for Exercise 3. Run this cell as is.
from collections import defaultdict


def _table(pairs):
    c = defaultdict(lambda: defaultdict(int))
    u = defaultdict(int)
    for w1, w2, n in pairs:
        c[w1][w2] += n
        u[w1] += n
    return c, u


def test_example():
    """from 'my', the model writes 'my aunt stayed quiet'"""
    out = generate("my", counts, unigram_counts)
    assert out == "my aunt stayed quiet", f"got '{out}'"


def test_picks_most_frequent_next_word():
    """it picks the next word that followed most often, not the first or last in the alphabet"""
    c, u = _table([("x", "m", 1), ("x", "z", 2), ("x", "a", 3)])
    out = generate("x", c, u)
    assert out != "x z", "it picked 'z', the last word in the alphabet. The key should be the count of each candidate"
    assert out != "x m", "it picked 'm', which followed x only once. The key should be the count of each candidate, candidates[w]"
    assert out == "x a", f"got '{out}', expected 'x a': a followed x 3 times"


def test_uses_the_pair_count():
    """it uses the count of the pair, not how often the word starts other pairs"""
    c, u = _table([("x", "b", 2), ("x", "a", 1), ("a", "c", 5)])
    out = generate("x", c, u)
    assert out.split()[:2] == ["x", "b"], (f"got '{out}'. 'a' starts many pairs, but after 'x' the word 'b' "
                                           "is more frequent. Use candidates[w], not unigram_counts[w]")


def test_adds_the_new_word():
    """each step adds the chosen word, and the next step starts from it"""
    c, u = _table([("p", "q", 1), ("q", "r", 1)])
    out = generate("p", c, u)
    assert out == "p q r", f"got '{out}', expected 'p q r'. Append the new word, not the start word"


def test_unseen_start_word():
    """a start word never seen as a first word gives just that word"""
    out = generate("her", counts, unigram_counts)
    assert out == "her", f"got '{out}'"


def test_stops_after_max_words():
    """a loop like 'a a a ...' stops after max_words new words"""
    c, u = _table([("a", "a", 1)])
    out = generate("a", c, u, max_words=5)
    assert len(out.split()) == 6, f"got {len(out.split())} words, expected 6: the start word and 5 new ones"
    assert generate("a", c, u, max_words=0) == "a", "with max_words=0, only the start word"

_failed = 0
for _test in [test_example, test_picks_most_frequent_next_word, test_uses_the_pair_count, test_adds_the_new_word, test_unseen_start_word, test_stops_after_max_words]:
    try:
        _test()
        print("✓", _test.__doc__)
    except Exception as _e:
        _failed += 1
        print("✗", _test.__doc__)
        print("   ", _e if isinstance(_e, AssertionError) else f"{type(_e).__name__}: {_e}")
assert _failed == 0, f"{_failed} test(s) failed: see the ✗ lines above"
print("✓ Exercise 3 passed")

<details><summary><b>Solution to Exercise 3</b> (open it after you have tried)</summary>

```python
def generate(start_word, counts, unigram_counts, max_words=10):
    word = start_word
    sentence = [word]
    for _ in range(max_words):
        candidates = counts[word]
        if not candidates:
            break
        word = max(candidates, key=lambda w: candidates[w])
        sentence.append(word)
    return " ".join(sentence)


print(generate("my", counts, unigram_counts))
```

</details>

**Reflection:** With only three sentences in the corpus, what happens if you run
`generate("her", counts, unigram_counts)`? Try it. Why does it behave that way, and what
does that tell you about what this model actually learned versus what it can generalize
to? Now think about Galician: there is far less raw Galician text sitting online than
Spanish or English text. Based on what you just saw, what would you predict about how an
n-gram model trained on real Galician text would behave compared to one trained on
English?

## Part B — Using n-grams to fix word order

A word-for-word translator turns "unha tía miña" into "a aunt my": the right words, in
the wrong order. A bigram model can score candidate word orders by how fluent
they sound and pick the best one. This section builds a small version of that.

### ✏️ Exercise 4 — Score word orders with a smoothed bigram model

Two gaps:

1. In `sentence_log_prob`, compute the **add-one smoothed** probability of each pair and
   add its logarithm to `log_prob`.
2. In the search loop, keep the best-scoring order seen so far.

<details><summary><b>Hints</b> (click to open)</summary>

- Add-one smoothing adds 1 to every pair count, and adds the vocabulary size to the denominator.
- Logs turn the product of probabilities into a sum: `log_prob += math.log(prob)`.
- Update the best order only when the new score is higher than `best_score`.

</details>

In [ ]:
import itertools
import math

FLUENCY_CORPUS = CORPUS + [
    "my aunt was very kind",
    "my aunt lived nearby",
    "an old friend of mine visited",
    "a friend of mine called",
]

fluency_counts, fluency_unigrams = build_bigram_counts(FLUENCY_CORPUS)
vocab_size = len(fluency_unigrams) + 10


def sentence_log_prob(words, counts, unigram_counts, vocab_size):
    tokens = ["<s>"] + words + ["</s>"]
    log_prob = 0.0
    for i in range(len(tokens) - 1):
        w1, w2 = tokens[i], tokens[i + 1]
        prob = (counts[w1][w2] + ___) / (unigram_counts[w1] + ___)
        log_prob += ___
    return log_prob


words_to_order = ["a", "aunt", "my"]

best_order, best_score = None, float("-inf")
for perm in itertools.permutations(words_to_order):
    score = sentence_log_prob(list(perm), fluency_counts, fluency_unigrams, vocab_size)
    print(f"{perm}  ->  {score:.3f}")
    if ___:
        best_score, best_order = ___, ___

print("\nBest order found:", " ".join(best_order))
print()
print("Better than the word-for-word order, purely from local fluency scoring — but still")
print("not quite right (that stray leftover \"a\" has nowhere grammatical to go). A permutation")
print("search over 3 words has no real grammar in it — it only knows which pairs of words")
print("tend to sit next to each other — never that a determiner attaches to a noun.")

In [ ]:
# Check for Exercise 4. Run this cell as is.
import math
from collections import defaultdict


def _table(pairs):
    c = defaultdict(lambda: defaultdict(int))
    u = defaultdict(int)
    for w1, w2, n in pairs:
        c[w1][w2] += n
        u[w1] += n
    return c, u


# my aunt: <s> -> my, my -> aunt, aunt -> </s>
_C, _U = _table([("my", "aunt", 2), ("my", "uncle", 1)])
_PROBS = [1 / 10, 3 / 13, 1 / 10]


def test_smoothed_formula():
    """each pair gets (count + 1) / (count of the first word + vocabulary size)"""
    got = sentence_log_prob(["my", "aunt"], _C, _U, 10)
    right = sum(math.log(p) for p in _PROBS)
    if abs(got - right) < 1e-9:
        return
    assert abs(got - sum(_PROBS)) > 1e-9, "log_prob adds the probabilities. Add their logarithms: math.log(prob)"
    assert abs(got - sum(math.log10(p) for p in _PROBS)) > 1e-9, "use math.log, the natural logarithm, not math.log10"
    assert got != 0, "log_prob stays 0. Add each term with +=, starting from 0.0"
    no_v = math.log(1 / 1) + math.log(3 / 4) + math.log(1 / 1)
    assert abs(got - no_v) > 1e-9, "the denominator should add vocab_size, not 1"
    assert False, f"got {got:.4f}, expected {right:.4f}"


def test_unseen_words_no_error():
    """an unseen word gets a small probability, not 0 and not an error"""
    got = sentence_log_prob(["zzz"], _C, _U, 10)
    assert abs(got - 2 * math.log(1 / 10)) < 1e-9, \
        "an unseen word makes two unseen pairs, each with probability 1 / vocab_size"


def test_empty_sentence():
    """an empty sentence is just <s> followed by </s>"""
    got = sentence_log_prob([], _C, _U, 10)
    assert abs(got - math.log(1 / 10)) < 1e-9, f"got {got:.4f}, expected log(1/10) = {math.log(1 / 10):.4f}"


def test_fluent_order_scores_higher():
    """'my aunt' scores higher than 'aunt my'"""
    v = vocab_size
    assert sentence_log_prob(["my", "aunt"], fluency_counts, fluency_unigrams, v) > \
        sentence_log_prob(["aunt", "my"], fluency_counts, fluency_unigrams, v), \
        "'my aunt' should score higher. Fix sentence_log_prob first: see the first test"


def test_search_keeps_the_best():
    """the search keeps the order with the highest score, and its score"""
    v = vocab_size
    score = lambda p: sentence_log_prob(list(p), fluency_counts, fluency_unigrams, v)
    best = max(itertools.permutations(words_to_order), key=score)
    worst = min(itertools.permutations(words_to_order), key=score)
    scores = {score(p) for p in itertools.permutations(words_to_order)}
    assert len(scores) > 1, "every order gets the same score. Fix sentence_log_prob first: see the first test"
    assert best_order is not None, "best_order is still None: the if never updated it"
    assert best_score != float("-inf"), "best_score is still -inf: update it together with best_order"
    assert tuple(best_order) != tuple(worst), "it kept the LOWEST score. Log probabilities: higher is better"
    assert tuple(best_order) == best, f"best_order is {tuple(best_order)}, expected {best}"
    assert abs(best_score - score(best)) < 1e-9, "best_score should be updated together with best_order"

_failed = 0
for _test in [test_smoothed_formula, test_unseen_words_no_error, test_empty_sentence, test_fluent_order_scores_higher, test_search_keeps_the_best]:
    try:
        _test()
        print("✓", _test.__doc__)
    except Exception as _e:
        _failed += 1
        print("✗", _test.__doc__)
        print("   ", _e if isinstance(_e, AssertionError) else f"{type(_e).__name__}: {_e}")
assert _failed == 0, f"{_failed} test(s) failed: see the ✗ lines above"
print("✓ Exercise 4 passed")

<details><summary><b>Solution to Exercise 4</b> (open it after you have tried)</summary>

```python
import itertools
import math

FLUENCY_CORPUS = CORPUS + [
    "my aunt was very kind",
    "my aunt lived nearby",
    "an old friend of mine visited",
    "a friend of mine called",
]

fluency_counts, fluency_unigrams = build_bigram_counts(FLUENCY_CORPUS)
vocab_size = len(fluency_unigrams) + 10


def sentence_log_prob(words, counts, unigram_counts, vocab_size):
    tokens = ["<s>"] + words + ["</s>"]
    log_prob = 0.0
    for i in range(len(tokens) - 1):
        w1, w2 = tokens[i], tokens[i + 1]
        prob = (counts[w1][w2] + 1) / (unigram_counts[w1] + vocab_size)
        log_prob += math.log(prob)
    return log_prob


words_to_order = ["a", "aunt", "my"]

best_order, best_score = None, float("-inf")
for perm in itertools.permutations(words_to_order):
    score = sentence_log_prob(list(perm), fluency_counts, fluency_unigrams, vocab_size)
    print(f"{perm}  ->  {score:.3f}")
    if score > best_score:
        best_score, best_order = score, perm

print("\nBest order found:", " ".join(best_order))
print()
print("Better than the word-for-word order, purely from local fluency scoring — but still")
print("not quite right (that stray leftover \"a\" has nowhere grammatical to go). A permutation")
print("search over 3 words has no real grammar in it — it only knows which pairs of words")
print("tend to sit next to each other — never that a determiner attaches to a noun.")
```

</details>

**Reflection:** How many orderings did the search above try, for 3 words? Work out
(or look up) how many orderings there would be for 8 words, and for a real 10-word
sentence. What kind of growth is that? Given that growth rate, could a system like this
one ever brute-force word order for a real sentence? What would a system need to do
instead?

### ✏️ Exercise 5 — Count the orderings, and predict the count

Count every ordering of `BIGGER_WORDS` in the loop, and compute `expected`, the number
of orderings, from a formula. The two numbers must match.

<details><summary><b>Hints</b> (click to open)</summary>

- n distinct words have n! orderings: n × (n − 1) × ... × 1.
- `math` was imported two cells above.

</details>

In [ ]:
!pip install -q codecarbon
from codecarbon import EmissionsTracker
import time

BIGGER_WORDS = ["a", "an", "aunt", "old", "friend", "of", "mine", "visited"]

tracker = EmissionsTracker(
    project_name="ngrams-permutation-search",
    measure_power_secs=1,
    save_to_file=False,
    log_level="warning",
)
tracker.start()
start_time = time.time()

count = 0
for perm in itertools.permutations(___):
    count += ___
expected = math.factorial(___)

elapsed = time.time() - start_time
emissions_kg = tracker.stop()

print(f"{count:,} orderings for {len(BIGGER_WORDS)} words, generated in {elapsed:.2f}s "
      f"({emissions_kg * 1000:.6f} g CO2eq)")
print("A real sentence has dozens of words. This is why nobody brute-forces word order at scale.")

In [ ]:
# Check for Exercise 5. Run this cell as is.
import math


def test_count():
    """the loop counts every ordering of BIGGER_WORDS once"""
    n = len(BIGGER_WORDS)
    assert count != 6, "count is 6: those are the orderings of the three words of Exercise 4. Loop over BIGGER_WORDS"
    assert count != n * math.factorial(n), "count adds the length of each ordering. Add 1 per ordering"
    assert count == 40320, f"count is {count}, expected 40,320"


def test_formula():
    """expected comes from the formula n!"""
    assert expected == math.factorial(len(BIGGER_WORDS)) == 40320, \
        f"expected is {expected}. n different words have n! orderings: math.factorial(len(BIGGER_WORDS))"


def test_match():
    """the counted and computed numbers match"""
    assert count == expected, f"count = {count}, expected = {expected}"

_failed = 0
for _test in [test_count, test_formula, test_match]:
    try:
        _test()
        print("✓", _test.__doc__)
    except Exception as _e:
        _failed += 1
        print("✗", _test.__doc__)
        print("   ", _e if isinstance(_e, AssertionError) else f"{type(_e).__name__}: {_e}")
assert _failed == 0, f"{_failed} test(s) failed: see the ✗ lines above"
print("✓ Exercise 5 passed")

<details><summary><b>Solution to Exercise 5</b> (open it after you have tried)</summary>

```python
!pip install -q codecarbon
from codecarbon import EmissionsTracker
import time

BIGGER_WORDS = ["a", "an", "aunt", "old", "friend", "of", "mine", "visited"]

tracker = EmissionsTracker(
    project_name="ngrams-permutation-search",
    measure_power_secs=1,
    save_to_file=False,
    log_level="warning",
)
tracker.start()
start_time = time.time()

count = 0
for perm in itertools.permutations(BIGGER_WORDS):
    count += 1
expected = math.factorial(len(BIGGER_WORDS))

elapsed = time.time() - start_time
emissions_kg = tracker.stop()

print(f"{count:,} orderings for {len(BIGGER_WORDS)} words, generated in {elapsed:.2f}s "
      f"({emissions_kg * 1000:.6f} g CO2eq)")
print("A real sentence has dozens of words. This is why nobody brute-forces word order at scale.")
```

</details>

## Code literacy check

Before you move on, pick at least two code cells above that you actually ran and read
today, and add `#` comments to them explaining, in your own words, what each meaningful
line or block does and *why* it's there, in language a classmate who hasn't seen the code
could follow.

Do this part on your own, after class:

1. Paste your commented code into an LLM chat (ChatGPT, Claude, Gemini — whichever you
   have access to).
2. Ask it something like: *"Here is code I commented myself, explaining what I think it
   does. Based only on my comments, how well do you think I understood this code? What
   did I get right, what did I miss, and what's still fuzzy?"*
3. Read what it says. Where it disagrees with your comment, go check the code again —
   who's right?

This is a self-check on your own understanding: a way to catch, in your own words, where
your mental model of the code breaks down against a second reader.